In [0]:
# Create struct schema for orders csv file
from pyspark.sql.types import *

order_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("promotion_id", IntegerType(), True)
    ])

In [0]:
# autoload csv into dataframe with schemalocation defined

df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option(
    "cloudFiles.schemaLocation",
    "/Volumes/first_data_engineering_project/pipeline_metadata/autoloader_metadata/schemas/bronze_orders") \
    .schema(order_schema) \
    .load("/Volumes/first_data_engineering_project/landing/retail_files/orders/") 

In [0]:
# add bronze layer metadata columns to the dataFrame for ingestion and lineage tracking
from pyspark.sql import functions as F
bronze_orders = df \
            .withColumn("ingestion_timestamp", F.current_timestamp()) \
            .withColumn("source_file", F.col("_metadata.file_name"))
            .withColumn( \
        "file_modified)time", 
        F.col("_metadata.file_modification_time")) 

In [0]:
# create table with checkpoint location and add trigger
bronze_orders.writeStream \
    .format("delta") \
    .option("checkpointLocation", 
            "/Volumes/first_data_engineering_project/pipeline_metadata/autoloader_metadata/checkpoints/bronze_orders") \
    .trigger(availableNow=True) \
    .toTable("first_data_engineering_project.bronze.bronze_orders")
